In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber
import extract_exercises_from_exam_file as parse

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [44]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [36]:
def classify_exercise(statement: str, topic: str, keywords = None) -> list:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topics = []
    for topic in keywords.keys():
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        if re.search(pattern, statement):
            topics.append(topic)

    return topics

In [45]:
def create_df_exercises_from_exam(exam_file: Path) -> pd.DataFrame:
    exercises_list = []

    # extraer el texto del examen
    exercises_text = parse.extract_exercises_from_pdf(exam_file)

    # clasificar el periodo del examen
    year = int(re.search(r"\d{4}", exam_file.name)[0])
    exam_period = parse.get_exam_period(year)

    # anadir los ejercicios a una lista de diccionarios para
    # luego crear el dataframe
    for ex in parse.split_exercises(exercises_text, exam_period):
        topic = classify_exercise(statement=ex, topic="termoquimica")
        exercises_list.append({"year" : year, "exam" : exam_file.name, "topics" : topic, "statement" : ex})
              
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de un archivo

In [49]:
file = ROOT / "examenes_quimica/2024_quimica/química - 2024 - junio.pdf"

In [18]:
lista_archivos_termoquimica = [
    # 2013 — Junio 6A, R1 4A, R2 5A, R3 5B, R4 5A, Septiembre 3B
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - junio.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - reserva 3.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - reserva 4.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2013_quimica/química - 2013 - septiembre.pdf",
    # 2014 — Junio 5A, R1 5B, R2 4A y 6B, R3 3B, R4 5A, Septiembre 6A
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - junio.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - reserva 3.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - reserva 4.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2014_quimica/química - 2014 - septiembre.pdf",
    # 2015 — Junio 6B, R1 3B, R2 6A, R3 5B, R4 5A, Septiembre 5A
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - junio.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - reserva 3.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - reserva 4.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2015_quimica/química - 2015 - septiembre.pdf",
    # 2016 — Junio 3B, R1 5A, R2 4A y 6B, R3 5B, Septiembre 5A
    "/home/daniel/code/selector_examenes/examenes_quimica/2016_quimica/2016_quimica_quimica - titular_junio_6 EXAMEN QUIMICA.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2016_quimica/química - 2016 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2016_quimica/química - 2016 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2016_quimica/química - 2016 - reserva 3.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2016_quimica/química - 2016 - septiembre.pdf",
    # 2024 — R1 C2, R2 B1, R3 C4, Julio B5
    "/home/daniel/code/selector_examenes/examenes_quimica/2024_quimica/química - 2024 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2024_quimica/química - 2024 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2024_quimica/química - 2024 - reserva 3.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2024_quimica/química - 2024 - julio.pdf",
    # 2025 — R1 3A, R2 5, Julio 2A y 5
    "/home/daniel/code/selector_examenes/examenes_quimica/2025_quimica/química - 2025 - reserva 1.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2025_quimica/química - 2025 - reserva 2.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2025_quimica/química - 2025 - julio.pdf",
    # 2026 — Junio 2A, Julio 2A (aún sin renombrar, pero el nombre oficial es inequívoco)
    "/home/daniel/code/selector_examenes/examenes_quimica/2026_quimica/2026_quimica_Quimica - Ordinaria_Titular - Examen Titular Quínica.pdf",
    "/home/daniel/code/selector_examenes/examenes_quimica/2026_quimica/2026_quimica_Quimica - Extraordinaria_Titular - Examen Titular Química_Ex.pdf",
]

In [50]:
create_df_exercises_from_exam(file)

,year,exam,topics,statement
0,2024,química - 2024 - junio.pdf,[formulacion],A1. Formule o nombre los siguientes compuestos...
1,2024,química - 2024 - junio.pdf,[formulacion],A2. Formule o nombre los siguientes compuestos...
2,2024,química - 2024 - junio.pdf,[configuracion_electronica],"B1. Dados los iones F⁻ y O²⁻, justifique la ve..."
3,2024,química - 2024 - junio.pdf,[],B2. El metanol se prepara industrialmente segú...
4,2024,química - 2024 - junio.pdf,[configuracion_electronica],B3. Dados tres elementos cuyas configuraciones...
5,2024,química - 2024 - junio.pdf,"[configuracion_electronica, propiedades_period...",B4. Conteste de forma razonada a las cuestione...
6,2024,química - 2024 - junio.pdf,[],B5. Justifique si las siguientes afirmaciones ...
7,2024,química - 2024 - junio.pdf,[],B6. Considerando los compuestos:\n(1) CH₃CHOHC...
8,2024,química - 2024 - junio.pdf,[],"C1. El N₂O₄ se descompone en NO₂, estableciénd..."
9,2024,química - 2024 - junio.pdf,[solubilidad],C2. Para preparar 250 mL de disolución saturad...


In [20]:
df_termo = pd.DataFrame()
for file in lista_archivos_termoquimica:
    file_path = Path(file)
    df_termo = pd.concat([df_termo, create_df_exercises_from_exam(file_path)], axis=0)

In [29]:
termo_exercises_df = df_termo[df_termo.topic =="termoquimica"]
termo_exercises_df["statement_short"] = termo_exercises_df.statement.apply(lambda x : x[:15]) 

In [31]:
termo_exercises_df.head()

,year,exam,topic,statement,statement_short
5,2013,química - 2013 - junio.pdf,termoquimica,"6, opción B En la reacción del oxígeno molecul...","6, opción B En"
3,2013,química - 2013 - reserva 1.pdf,termoquimica,"4, opción A a) La entalpía de formación del NH...","4, opción A a)"
11,2013,química - 2013 - reserva 1.pdf,termoquimica,"6, opción B a) Establezca el ciclo termoquímic...","6, opción B a)"
4,2013,química - 2013 - reserva 2.pdf,termoquimica,"5, opción A Tanto el etanol (C₂H₅OH) como la g...","5, opción A Tan"
3,2013,química - 2013 - reserva 3.pdf,termoquimica,"4, opción A Dado el compuesto HOCH₂CH₂CH₂CH=CH...","4, opción A Dad"


In [33]:
termo_exercises_df[["exam", "topic", "statement_short"]].to_csv("check_termo.csv", index = False)

In [17]:
with open("examenes_termoquimica.txt", "w") as file:
    file.writelines("\n".join(lista_archivos_termoquimica))

In [35]:
for idx, row in termo_exercises_df[termo_exercises_df.year == 2013].iterrows():
    print(row.exam)
    print(row.statement)
    print("\n\n")

química - 2013 - junio.pdf
6, opción B En la reacción del oxígeno molecular gaseoso con el cobre para formar óxido de cobre(II) se
desprenden 2,30 kJ por cada gramo de cobre que reacciona, a 298 K y 760 mm Hg. Calcule:
a) La entalpía de formación del óxido de cobre(II).
b) El calor desprendido a presión constante cuando reaccionan 100 L de oxígeno, medidos a 1,5 atm y
27ºC.
Datos: R = 0,082 atm·L·moI⁻¹·K⁻¹; Masa atómica Cu = 63,5.

Opción B





química - 2013 - reserva 1.pdf
4, opción A a) La entalpía de formación del NH₃(g) a 298 K es Δ = 46,11 kJ/mol. Escriba la ecuación química
a la que se refiere este valor.
b) ¿Cuál es la variación de energía interna (ΔU) de un sistema si absorbe un calor de 67 J y realiza un
trabajo de 67 J? Razone la respuesta.
c) ¿Puede una reacción exotérmica no ser espontánea? Razone la respuesta.





química - 2013 - reserva 1.pdf
6, opción B a) Establezca el ciclo termoquímico de Born-Haber para la formación de CaCl₂(s).
b) Calcule la afinidad electrónica